# Прогнозирование фактического времени доставки еды

**Кейс №5 (Проектный практикум-1). Регрессия: фактическое время доставки.**

**Задача.** Спрогнозировать точное количество минут до завершения доставки **с момента размещения заказа** на основе времени приготовления блюд, дистанции, категории пробок, погоды и характеристик курьера.

**Формализация:**
- Целевая переменная — `actual_delivery_time_minutes` (непрерывная величина, минуты) → задача **регрессии**.
- Один объект в данных — **один заказ** (каждая строка датасета уникальна по `order_id`).
- Прогноз должен быть доступен в момент заказа, поэтому признаками используем только ту информацию, которая известна в момент размещения заказа (это критично для исключения утечек данных — см. раздел о валидации).

| Участник | Вклад |
|---|---|
| Никита | EDA: технический паспорт данных, структура пропусков, сегментный анализ, корреляции (PHIK), план признаков |
| Дмитрий | Очистка данных, baseline-модель линейной регрессии, масштабирование, feature engineering (взаимодействия), диагностика остатков |
| Любовь | EDA: визуализация целевой переменной, анализ аномалий (факт < времени готовки), анализ причин отмен заказов |
| Егор | EDA: первичный отбор признаков, рейтинговая таблица корреляций с таргетом, временные фичи (часы пик); baseline: бизнес-baseline «текущая ETA платформы», анализ доли опозданий, разбор экстремальных заказов |

Сборка единого ноутбука: вся команда. Все `random_state` зафиксированы константой `RANDOM_STATE = 42`; ноутбук выполняется сверху вниз без вмешательств.

**Заметка от PM о том, как была организована работа:**
По своей сути, все доступные члены команды отправлялись на решение текущей, актуальной задачи (по началу EDA, потом baseline...), что в итоге своём давало множество результатов, в целом схожих, но различающихся в деталях. 
Такой подход дал больше данных, подтверждений выводов, гипотез, самые удачные из которых затем были объединены в данный ноутбук.



## 1. Загрузка данных и технический паспорт

Данные: `food_delivery_orders_dataset.csv` (50 000 заказов, 37 столбцов).



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', None)

DATA_PATH = next(p for p in [
    Path('../Data/food_delivery_orders_dataset.csv'),
    Path('../../Data/food_delivery_orders_dataset.csv'),
    Path('Data/food_delivery_orders_dataset.csv'),
] if p.exists())
print('Данные:', DATA_PATH)
df = pd.read_csv(DATA_PATH)
print(f'Размер датасета: {df.shape[0]} строк x {df.shape[1]} столбцов')
df.head(3)



Данные: ..\..\Data\food_delivery_orders_dataset.csv


Размер датасета: 50000 строк x 37 столбцов


,order_id,customer_id,restaurant_id,driver_id,order_timestamp,order_date,order_hour,day_of_week,is_weekend,city,delivery_area,customer_age,customer_type,restaurant_type,restaurant_primary_category,restaurant_rating,items_count,subtotal,discount_percent,tax_amount,service_fee,delivery_fee,order_total,payment_method,tip_amount,distance_km,weather,traffic_level,delivery_partner_experience_months,delivery_partner_rating,restaurant_preparation_time_minutes,estimated_delivery_time_minutes,actual_delivery_time_minutes,late_delivery,order_status,cancellation_reason,customer_rating
0,ORD_000001,CUS_00001,RES_0008,DRV_0075,2025-10-05 15:54:00,2025-10-05,15,Sunday,1,City_C,City_C_Area_3,40,Premium,Casual Dining,Healthy,4.4,4,64.82,0,9.07,2.40,0.00,76.29,Credit Card,11.36,1.1,Rain,Medium,14,4.4,25,25,29.0,0.0,Completed,NaN,4.6
1,ORD_000002,CUS_00013,RES_0024,DRV_0131,2025-10-01 20:00:00,2025-10-01,20,Wednesday,0,City_A,City_A_Area_1,19,Regular,Cafe,Dessert,4.4,2,13.03,0,1.82,0.84,3.25,18.94,Cash,1.09,2.5,Cloudy,High,44,4.7,12,20,24.0,0.0,Completed,NaN,5.0
2,ORD_000003,CUS_00049,RES_0013,DRV_0087,2025-10-07 18:24:00,2025-10-07,18,Tuesday,0,City_C,City_C_Area_3,26,Regular,Fast Food,Dessert,4.8,1,10.12,10,1.28,0.56,7.90,18.85,Cash,0.00,11.8,Cloudy,High,1,5.0,11,63,72.0,1.0,Completed,NaN,3.6


In [2]:
# Технический паспорт: типы, диапазоны
info = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(),
    'nulls': df.isna().sum(),
})
display(info)



,dtype,n_unique,nulls
order_id,object,50000,0
customer_id,object,4979,0
restaurant_id,object,250,0
driver_id,object,200,0
order_timestamp,object,25459,0
order_date,object,30,0
order_hour,int64,24,0
day_of_week,object,7,0
is_weekend,int64,2,0
city,object,4,0


### 1.1 Смотрим пропуски



In [3]:
nulls = df.isna().sum()
print('Пропуски по столбцам (только ненулевые):')
print(nulls[nulls > 0].sort_values(ascending=False))
print()
print('Доля пропусков ключевых полей по статусу заказа:')
cols = ['actual_delivery_time_minutes', 'estimated_delivery_time_minutes',
        'late_delivery', 'customer_rating', 'cancellation_reason']
print(df.groupby('order_status')[cols].apply(lambda g: g.isna().mean().round(3)).T)



Пропуски по столбцам (только ненулевые):
cancellation_reason             49132
tip_amount                        868
actual_delivery_time_minutes      868
late_delivery                     868
customer_rating                   868
dtype: int64

Доля пропусков ключевых полей по статусу заказа:
order_status                     Cancelled  Completed
actual_delivery_time_minutes           1.0        0.0
estimated_delivery_time_minutes        0.0        0.0
late_delivery                          1.0        0.0
customer_rating                        1.0        0.0
cancellation_reason                    0.0        1.0


**Вывод.** Все пропуски сосредоточены в полях, осмысленных только для одного статуса: у `Cancelled` нет фактического времени доставки и оценки, у `Completed` — нет причины отмены. Это **структура данных**, а не ошибки сбора: в итоговый набор для регрессии мы возьмём только `Completed`, и пропуски в целевой переменной исчезнут сами. `cancellation_reason` (98% пропусков) не является признаком для прогноза — не используем.

Дубликаты проверяем отдельно:


In [4]:
print('Полных дубликатов строк:', df.duplicated().sum())
print('Дубликатов order_id:', df['order_id'].duplicated().sum())



Полных дубликатов строк: 0
Дубликатов order_id: 0


### 1.2 Санити-чек целевой переменной и скорости курьеров

Проверяем физическую правдоподобность: чистое время в пути = факт − время готовки; скорость = дистанция / время в пути.



In [5]:
d = df[df['order_status'] == 'Completed'].copy()
d['transit_min'] = d['actual_delivery_time_minutes'] - d['restaurant_preparation_time_minutes']
d['delay'] = d['actual_delivery_time_minutes'] - d['estimated_delivery_time_minutes']
d['speed_kmh'] = d['distance_km'] / d['transit_min'] * 60

print('Заказов Completed:', len(d))
print('Заказов с transit <= 0 (факт <= времени готовки):', (d['transit_min'] <= 0).sum())
print()
print('Скорость (после исключения артефактов):')
print(d.loc[d['transit_min'] > 0, 'speed_kmh'].describe().round(2))
print()
print('Скорость по погоде (медиана):')
print(d.loc[d['transit_min'] > 0].groupby('weather')['speed_kmh'].median().round(1))



Заказов Completed: 49132
Заказов с transit <= 0 (факт <= времени готовки): 74

Скорость (после исключения артефактов):
count    49058.00
mean        15.92
std          6.75
min          4.29
25%         12.12
50%         14.69
75%         18.00
max        168.00
Name: speed_kmh, dtype: float64

Скорость по погоде (медиана):
weather
Clear     15.8
Cloudy    14.4
Rain      12.0
Storm      9.3
Name: speed_kmh, dtype: float64


**Вывод.**
- Медианная скорость ~15 км/ч — реалистично для города (велосипед/скутер/пробки); в шторм скорость падает — косвенное подтверждение, что погода влияет и на «внешние» минуты.
- 74 заказа (<0.2%) с `transit ≤ 0` — логическая аномалия (факт меньше времени готовки — артефакт генерации данных). Это правило физической невозможности, а не статистика, посчитанная по данным, поэтому удаление таких строк **не создаёт утечки**. Для моделирования исключаем.
- Экстремально долгие доставки (до ~172 мин) физически возможны (дальние заказы в шторм) — **выбросы сохраняем**: это и есть «проблемные кейсы», ради которых строится прогноз.



## 2. Исследовательский анализ (EDA)


### 2.1 Целевая переменная: факт vs прогноз платформы



In [6]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Фактическое время доставки (мин)', 'Прогноз платформы (мин)'))
fig.add_trace(go.Histogram(x=d['actual_delivery_time_minutes'], nbinsx=60,
                           marker_color='#4c78a8', name='факт'), row=1, col=1)
fig.add_trace(go.Histogram(x=d['estimated_delivery_time_minutes'], nbinsx=60,
                           marker_color='#f58518', name='прогноз'), row=1, col=2)
fig.update_layout(showlegend=False, height=380, title='Распределения: оба скошены вправо')
fig.show()

delay = d['actual_delivery_time_minutes'] - d['estimated_delivery_time_minutes']
print('Среднее (факт - прогноз):', round(delay.mean(), 2), 'мин')
print('Доля заказов, где факт больше прогноза:', round((delay > 0).mean() * 100, 1), '%')



Среднее (факт - прогноз): 4.23 мин
Доля заказов, где факт больше прогноза: 75.9 %


**Вывод.** Оба распределения скошены вправо: долгие доставки редки, но протяжённы. Прогноз платформы **систематически занижает** время — клиенты в среднем ждут дольше, чем им обещали. Наша задача — модель, которая предсказывает факт точнее, чем встроенный прогноз.

### 2.2 Внешние условия: погода и пробки



In [7]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Погода', 'Пробки'))
w_order = ['Clear', 'Cloudy', 'Rain', 'Storm']
t_order = ['Low', 'Medium', 'High']
for i, col in enumerate(['weather', 'traffic_level'], start=1):
    fig.add_trace(go.Box(y=d['actual_delivery_time_minutes'],
                         x=d[col].astype(str),
                         marker_color='#4c78a8', showlegend=False), row=1, col=i)
fig.update_layout(height=400, title='Время доставки по внешним условиям')
fig.show()

print(d.groupby('weather')['actual_delivery_time_minutes'].mean().reindex(w_order).round(1))
print()
print(d.groupby('traffic_level')['actual_delivery_time_minutes'].mean().reindex(t_order).round(1))
print()
# Взаимодействие погода x пробки: средняя ошибка прогноза платформы
pv = d.pivot_table(index='weather', columns='traffic_level',
                   values='delay', observed=True, aggfunc='mean').reindex(index=w_order, columns=t_order)
print('Средний остаток (факт − прогноз платформы), мин:')
print(pv.round(1))



weather
Clear     33.6
Cloudy    35.1
Rain      39.2
Storm     47.1
Name: actual_delivery_time_minutes, dtype: float64

traffic_level
Low       30.6
Medium    35.0
High      39.9
Name: actual_delivery_time_minutes, dtype: float64

Средний остаток (факт − прогноз платформы), мин:
traffic_level   Low  Medium  High
weather                          
Clear           2.5     2.6   2.5
Cloudy          3.9     4.3   4.7
Rain            7.2     8.3  10.1
Storm          12.5    15.9  18.6


**Вывод.** Среднее время монотонно растёт и по погоде (Clear → Storm), и по пробкам (Low → High). Ключевой инсайт — **взаимодействие**: при ясной погоде пробки почти не меняют ошибку платформы, но в дождь/шторм высокая пробковая нагрузка резко увеличивает недооценку. Это прямая мотивация для признаков-взаимодействий `дистанция × пробки` и `дистанция × погода` в моделировании (линейная модель со сложением признаков такой эффект не улавливает).


### 2.3 Опоздания и экстремальные заказы

Отдельный взгляд на «проблемные» заказы: как часто платформа опаздывает и как выглядят самые долгие доставки.

In [8]:
# Доля опозданий платформы и её связь с погодой (анализ из baseline Егора)
print('Доля опозданий (факт > прогноз платформы):',
      round((d['delay'] > 0).mean() * 100, 1), '%')
print()
print('Доля опозданий по погоде, %:')
print((d.groupby('weather')['delay'].apply(lambda s: (s > 0).mean()) * 100)
      .reindex(w_order).round(1))
print()
# Разбор самых долгих доставок
top = d.nlargest(3, 'actual_delivery_time_minutes')[
    ['actual_delivery_time_minutes', 'estimated_delivery_time_minutes',
     'restaurant_preparation_time_minutes', 'distance_km', 'weather',
     'traffic_level', 'delay']]
print('Топ-3 самых долгих доставки:')
display(top)


Доля опозданий (факт > прогноз платформы): 75.9 %

Доля опозданий по погоде, %:
weather
Clear     68.1
Cloudy    80.4
Rain      94.1
Storm     98.1
Name: delay, dtype: float64

Топ-3 самых долгих доставки:


,actual_delivery_time_minutes,estimated_delivery_time_minutes,restaurant_preparation_time_minutes,distance_km,weather,traffic_level,delay
4961,172.0,97,7,19.7,Storm,High,75.0
19908,158.0,96,22,17.3,Storm,High,62.0
35011,156.0,88,11,16.9,Storm,High,68.0


**Вывод.** Опоздания — не хвостовая экзотика, а норма для текущего прогноза платформы: факт больше обещанного в ~76% заказов, и доля опозданий монотонно растёт с ухудшением погоды (68% при ясной погоде → 98% в шторм). Самые долгие доставки (100+ минут) — это комбинация «большая дистанция + шторм + высокие пробки», при этом платформа недооценивает их сильнее всего. Это подтверждает: длинный хвост таргета — реальные кейсы, а не мусор, поэтому выбросы мы сохраняем, а MAE как метрику выбираем осознанно (см. раздел 4).

### 2.4 Дистанция, время готовки и временные закономерности



In [9]:
d['dist_bin'] = pd.cut(d['distance_km'], [0, 2, 5, 8, 11, 26],
                       labels=['0–2', '2–5', '5–8', '8–11', '11+'])
seg = d.groupby('dist_bin', observed=True).agg(
    n=('delay', 'size'),
    mean_actual=('actual_delivery_time_minutes', 'mean'),
    mean_prep=('restaurant_preparation_time_minutes', 'mean'),
    late_rate=('delay', lambda s: (s > 0).mean()),
)
print(seg.round(2))

fig = make_subplots(specs=[[{'secondary_y': True}]])
h = d.groupby('order_hour').agg(n=('delay', 'size'),
                                mean_actual=('actual_delivery_time_minutes', 'mean')).reset_index()
fig.add_bar(x=h['order_hour'], y=h['n'], name='заказов', marker_color='#B6B6F5', opacity=0.5)
fig.add_scatter(x=h['order_hour'], y=h['mean_actual'], name='среднее время, мин',
                line=dict(color='#d62728', width=3), secondary_y=True)
fig.update_layout(height=400, title='Заказы по часам и среднее время доставки')
fig.show()



              n  mean_actual  mean_prep  late_rate
dist_bin                                          
0–2        5267        22.59      15.58       0.73
2–5       26818        30.07      15.54       0.75
5–8       11711        41.20      15.44       0.78
8–11       3558        53.68      15.76       0.79
11+        1778        72.09      15.69       0.81


**Вывод.** `distance_km` — главный внешний драйвер: среднее время растёт монотонно от коротких до дальних заказов, доля «опозданий» платформы тоже растёт. Час заказа влияет слабо (в пики спроса время чуть выше, но остаток платформы от часа почти не зависит — прогноз уже адаптирован к нагрузке), поэтому час оставим как числовой признак без больших ожиданий.

### 2.4 Курьеры: есть ли эффект после контроля расстояния и пробок



In [10]:
d['exp_bin'] = pd.cut(d['delivery_partner_experience_months'], [-1, 6, 18, 36, 61],
                      labels=['<6', '6–18', '18–36', '36+'])
print(d.groupby('exp_bin', observed=True).agg(
    n=('delay', 'size'),
    mean_actual=('actual_delivery_time_minutes', 'mean'),
    mean_dist=('distance_km', 'mean'),
).round(2))
print()
print('Корреляция времени с опытом курьера:',
      round(d['actual_delivery_time_minutes'].corr(d['delivery_partner_experience_months']), 3))
print('Корреляция времени с рейтингом курьера:',
      round(d['actual_delivery_time_minutes'].corr(d['delivery_partner_rating']), 3))



             n  mean_actual  mean_dist
exp_bin                               
<6        5828        34.96       4.73
6–18      8226        35.27       4.78
18–36    16876        35.21       4.73
36+      18202        35.10       4.70

Корреляция времени с опытом курьера: -0.001
Корреляция времени с рейтингом курьера: 0.002


**Вывод.** Различия по группам опыта малы и во многом объясняются разным средним расстоянием групп; парные корреляции с таргетом почти нулевые. Гипотеза «опытный курьер едет быстрее» в этих данных практически не подтверждается — признаки курьера оставим в модели, но ожидаем у них малые веса (baseline-ноутбук Дмитрия это уже показал).


### 2.6 Корреляции: числовые + PHIK



In [11]:
from phik import phik_matrix

core_cols = ['actual_delivery_time_minutes', 'estimated_delivery_time_minutes',
             'restaurant_preparation_time_minutes', 'distance_km', 'items_count',
             'order_hour', 'delivery_partner_experience_months',
             'delivery_partner_rating', 'is_weekend',
             'weather', 'traffic_level', 'city', 'restaurant_type']
sample = d[core_cols].sample(8000, random_state=RANDOM_STATE)

interval_cols = ['actual_delivery_time_minutes', 'estimated_delivery_time_minutes',
                 'restaurant_preparation_time_minutes', 'distance_km', 'items_count',
                 'order_hour', 'delivery_partner_experience_months', 'delivery_partner_rating']
phi = phik_matrix(sample, interval_cols=interval_cols)

hm = go.Figure(go.Heatmap(z=phi.values, x=phi.columns, y=phi.columns,
                          zmin=0, zmax=1, colorscale='Blues',
                          text=np.round(phi.values, 2), texttemplate='%{text}'))
hm.update_layout(height=650, width=800, title='PHIK-корреляции (выборка 8000 заказов)')
hm.show()

print('PHIK с целевой переменной:')
print(phi['actual_delivery_time_minutes'].drop('actual_delivery_time_minutes').sort_values(ascending=False).round(3))



PHIK с целевой переменной:
estimated_delivery_time_minutes        0.920
distance_km                            0.799
restaurant_preparation_time_minutes    0.565
items_count                            0.414
traffic_level                          0.275
weather                                0.211
restaurant_type                        0.182
order_hour                             0.083
delivery_partner_experience_months     0.066
is_weekend                             0.060
city                                   0.007
delivery_partner_rating                0.000
Name: actual_delivery_time_minutes, dtype: float64


**Выводы по корреляциям.**
- Таргет определяется **дистанцией, временем готовки и прогнозом платформы**. `estimated` — производная платформы от тех же величин, поэтому его связь с фактом близка к 1: это «baseline-подобная» связь, а не мультиколлинеарность. Мы строим **сценарий «без estimated»** — честный прогноз «с нуля», как требует кейс (`estimated` технически известен в момент заказа, но добавлять его — значит «подгонять» модель к готовому ответу платформы).
- Из анализа исключены деньги (subtotal, tip), `late_delivery`, `customer_rating` — они либо известны после доставки, либо не несут причинной связи с временем; их использование — классическая утечка.
- `delivery_area` (40 категорий, часть групп мала) решено исключить: безопасное кодирование потребовало бы target encoding внутри CV, а выигрыш от города (`city`, 3 категории) и дистанции перекрывается имеющимися признаками.



## 3. Итоги EDA и стратегия предобработки

1. **Целевая переменная**: `actual_delivery_time_minutes`, объект — заказ, задача — регрессия.
2. **Очистка**: берём только `order_status == 'Completed'` (у остальных нет таргета); удаляем строки с `transit ≤ 0` (физическая аномалия); дубликатов нет.
3. **Пропуски**: структурные (по статусу) — после фильтрации исчезают; `cancellation_reason` и все «денежные»/после-доставочные поля не используем (утечка); кроме того, финансовые поля (`subtotal`, `order_total`, `tax_amount`, `service_fee`) сильно мультиколлинеарны между собой (r ≈ 0.9–1.0 — из корреляционной матрицы baseline-ноутбука Егора), так что даже вне вопроса утечки из этой группы достаточно одного признака, а для прогноза «с нуля» они не нужны вовсе.
4. **Признаки**: числовые — `distance_km`, `restaurant_preparation_time_minutes`, `items_count`, `order_hour`, `delivery_partner_experience_months`, `delivery_partner_rating`, `is_weekend`; категориальные — `weather`, `traffic_level`, `city`, `restaurant_type` → **One-Hot Encoding** (категории номинальные; древовидной модели кодирование вообще не нужно, линейной — именно OHE, а не порядковое: разница «Rain − Clear» не равна «Storm − Rain»). Дистанция и время готовки — в разных единицах (км/минуты), поэтому вопрос масштабирования решаем экспериментально (см. ★-раздел).
5. **Feature engineering-гипотезы** (проверим в моделировании): взаимодействия `distance × traffic`, `distance × weather` (из п. 2.2) и `is_rush` (обед/вечер, из п. 2.4).
6. **Схема валидации**: 5-fold KFold (shuffle, seed=42) на train; holdout 20% — только для финальной оценки. Вся предобработка — внутри Pipeline, чтобы scaler/OHE обучались только на train-фолдах.



## 4. Данные для моделирования и выбор метрик



In [12]:
from sklearn.model_selection import train_test_split

NUM_BASE = ['distance_km', 'restaurant_preparation_time_minutes', 'items_count',
            'order_hour', 'delivery_partner_experience_months',
            'delivery_partner_rating', 'is_weekend']
CAT_BASE = ['weather', 'traffic_level', 'city', 'restaurant_type']

df_model = d.loc[d['transit_min'] > 0, NUM_BASE + CAT_BASE +
                 ['actual_delivery_time_minutes', 'estimated_delivery_time_minutes']].copy()
df_model['is_weekend'] = df_model['is_weekend'].astype(int)

# estimated нужен только для бизнес-baseline (раздел 5) — в модели он не участвует:
# ColumnTransformer выбирает строго NUM_BASE + CAT_BASE
X = df_model[NUM_BASE + CAT_BASE + ['estimated_delivery_time_minutes']]
y = df_model['actual_delivery_time_minutes']

# Holdout откладываем сразу и до конца его не трогаем — только финальная оценка
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f'train: {X_train.shape[0]},  test: {X_test.shape[0]}')
print(f'Средний таргет train: {y_train.mean():.1f} мин, test: {y_test.mean():.1f} мин')



train: 39246,  test: 9812
Средний таргет train: 35.2 мин, test: 35.1 мин


**Выбор метрик.**
- **Основная — MAE** (средняя абсолютная ошибка, в минутах): (а) интерпретируема для бизнеса — «в среднем промахиваемся на N минут»; (б) устойчева к длинному правому хвосту таргета — редкие сверхдолгие доставки не «затирают» качество на типичных заказах; (в) одинаковый штраф за пере- и недооценку, что честно для обещаний клиентам.
- **Вспомогательные**: RMSE — штрафует крупные промахи (для SLA важно не допускать ошибок в 40+ минут); R² — доля объяснённой дисперсии для сравнения с baseline платформы. Распределение ошибок смотрим дополнительно (квантиль 90/95%), т.к. MAE может маскировать тяжёлый хвост.


## 5. Схема валидации и baseline

Валидация: **5-fold KFold (shuffle, random_state=42)** на train; результат — **среднее ± std по фолдам**, не одна цифра. Вся предобработка внутри `Pipeline`/`ColumnTransformer` — scaler и OHE обучаются только на train-части каждого фолда, поэтому утечки исключены по построению.



In [13]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, OrdinalEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
from sklearn.base import BaseEstimator, RegressorMixin


class PlatformETA(BaseEstimator, RegressorMixin):
    """«Модель», возвращающая готовый прогноз платформы — бизнес-baseline.

    Ничего не учит: на train она обязана показать MAE платформы на тех же фолдах,
    что и наши модели, чтобы сравнение было корректным (идея из baseline-ноутбука Егора).
    """

    def fit(self, X, y):
        return self

    def predict(self, X):
        return X['estimated_delivery_time_minutes'].values

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = ['neg_mean_absolute_error', 'neg_root_mean_squared_error', 'r2']

def make_preprocessor(scaler=None):
    num_step = ('num', scaler, NUM_BASE) if scaler is not None else ('num', 'passthrough', NUM_BASE)
    return ColumnTransformer([
        num_step,
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_BASE),
    ])

def cv_report(name, model, X_data, y_data, cv_split=cv, scoring=SCORING):
    res = cross_validate(model, X_data, y_data, cv=cv_split, scoring=scoring, n_jobs=-1)
    return {
        'model': name,
        'MAE': -res['test_neg_mean_absolute_error'].mean(),
        'MAE_std': res['test_neg_mean_absolute_error'].std(),
        'RMSE': -res['test_neg_root_mean_squared_error'].mean(),
        'R2': res['test_r2'].mean(),
    }

baseline_pipe = Pipeline([
    ('prep', make_preprocessor()),
    ('model', DummyRegressor(strategy='median')),
])
linear_pipe = Pipeline([
    ('prep', make_preprocessor()),
    ('model', LinearRegression()),
])

rows = [cv_report('Dummy (median)', baseline_pipe, X_train, y_train),
        cv_report('LinearRegression', linear_pipe, X_train, y_train),
        cv_report('Текущая ETA платформы (бизнес-baseline)',
                  Pipeline([('model', PlatformETA())]), X_train, y_train)]
pd.DataFrame(rows).round(3)



,model,MAE,MAE_std,RMSE,R2
0,Dummy (median),10.792,0.038,14.843,-0.022
1,LinearRegression,2.758,0.025,3.748,0.935
2,Текущая ETA платформы (бизнес-baseline),5.242,0.036,6.912,0.778


**Вывод.** `DummyRegressor` фиксирует нижнюю границу: модель обязана побить «предсказание медианой». Третья строка — **бизнес-baseline**: готовый прогноз платформы, оценённый на тех же фолдах, что и наши модели (идея из baseline-ноутбука Егора: «если модель не лучше текущей ETA — бизнес-смысла в ней нет»). Линейная регрессия на базовых признаках уже снижает MAE в разы и обгоняет ETA платформы — отправная точка для улучшений зафиксирована. Все модели имеют одинаковую схему валидации — результаты по фолдам сопоставимы и воспроизводимы.

## 6. ★ Сравнение способов масштабирования

Тестируем на **одной и той же схеме валидации** (те же фолды, тот же Pipeline): без масштабирования, StandardScaler, RobustScaler, MinMaxScaler. Смотрим и на метрику, и на коэффициенты линейной модели (число значимых и их суммарная величина).



In [14]:
from sklearn.preprocessing import RobustScaler, MinMaxScaler

scalers = {
    'Без масштабирования': None,
    'StandardScaler': StandardScaler(),
    'RobustScaler': RobustScaler(),
    'MinMaxScaler': MinMaxScaler(),
}
scale_rows, coef_info = [], {}
for name, scaler in scalers.items():
    pipe = Pipeline([('prep', make_preprocessor(scaler)),
                     ('model', LinearRegression())])
    scale_rows.append(cv_report(name, pipe, X_train, y_train))
    pipe.fit(X_train, y_train)
    coef_info[name] = pipe.named_steps['model'].coef_

scale_df = pd.DataFrame(scale_rows).round(4)
display(scale_df)

feat_names = pipe.named_steps['prep'].get_feature_names_out()
coef_df = pd.DataFrame({k: np.round(v, 3) for k, v in coef_info.items()}, index=feat_names)
display(coef_df.reindex(coef_df['StandardScaler'].abs().sort_values(ascending=False).index).head(12))
print('Сумма |коэффициентов| по вариантам:')
print(pd.Series({k: np.abs(v).sum() for k, v in coef_info.items()}).round(2))
print('Число коэффициентов с |w| > 0.01 (Standard):',
      int((np.abs(coef_info['StandardScaler']) > 0.01).sum()))



,model,MAE,MAE_std,RMSE,R2
0,Без масштабирования,2.7582,0.0255,3.7476,0.9348
1,StandardScaler,2.7582,0.0255,3.7476,0.9348
2,RobustScaler,2.7582,0.0255,3.7476,0.9348
3,MinMaxScaler,2.7582,0.0255,3.7476,0.9348


,Без масштабирования,StandardScaler,RobustScaler,MinMaxScaler
num__distance_km,4.047,11.223,12.547,77.710
cat__weather_Storm,7.617,7.617,7.617,7.617
num__restaurant_preparation_time_minutes,1.000,7.327,8.996,67.972
cat__weather_Clear,-5.106,-5.106,-5.106,-5.106
cat__traffic_level_High,4.839,4.839,4.839,4.839
cat__traffic_level_Low,-4.584,-4.584,-4.584,-4.584
cat__weather_Cloudy,-3.395,-3.395,-3.395,-3.395
cat__weather_Rain,0.885,0.885,0.885,0.885
cat__traffic_level_Medium,-0.255,-0.255,-0.255,-0.255
cat__city_City_B,0.073,0.073,0.073,0.073


Сумма |коэффициентов| по вариантам:
Без масштабирования     32.05
StandardScaler          45.54
RobustScaler            48.55
MinMaxScaler           172.78
dtype: float64
Число коэффициентов с |w| > 0.01 (Standard): 19


**Вывод.** Для линейной регрессии без регуляризации масштабирование **не меняет предсказания** (линейное преобразование признаков компенсируется коэффициентами): метрики совпадают до четвёртого знака — это ожидаемый теоретический результат, а не случайность. Разница только в **интерпретации**: после StandardScaler веса становятся сравнимыми между собой (разные единицы — км и минуты — приведены к единой шкале), и по ним видна реальная важность признаков: доминируют дистанция, время готовки и dummy-признаки пробок/погоды; признаки курьера — около нуля, что подтверждает выводы EDA (п. 2.5). **Оставляем StandardScaler**: он не ухудшает качество, делает коэффициенты сопоставимыми и обязателен для корректной работы регуляризации (Ridge/Lasso в следующем разделе). Для древовидных моделей масштабирование не нужно в принципе — деревья инвариантны к монотонным преобразованиям признаков.


## 7. Feature engineering: проверяемые гипотезы

Три гипотезы из EDA, каждая проверяется на одной и той же CV-схеме, дельта метрики фиксируется:
1. `distance × traffic_num`, `distance × weather_num` — из п. 2.2 (минута пробок «дороже» на больших дистанциях);
2. `is_rush` (обед 12-14, вечер 18-21, часы пик) — из п. 2.3;
3. `prep × items_count` — больше позиций -> дольше готовка «сверх» базового времени.



In [15]:
def add_features(df_part):
    res = df_part.copy()
    traffic_num = res['traffic_level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    weather_num = res['weather'].map({'Clear': 0, 'Cloudy': 1, 'Rain': 2, 'Storm': 3})
    res['dist_x_traffic'] = res['distance_km'] * traffic_num
    res['dist_x_weather'] = res['distance_km'] * weather_num
    res['is_rush'] = res['order_hour'].isin([12, 13, 14, 18, 19, 20, 21]).astype(int)
    res['prep_x_items'] = res['restaurant_preparation_time_minutes'] * res['items_count']
    return res

NUM_FE = NUM_BASE + ['dist_x_traffic', 'dist_x_weather', 'is_rush', 'prep_x_items']
CAT_FE = CAT_BASE

def make_pipe_fe(scaler=StandardScaler(), model=LinearRegression()):
    return Pipeline([
        ('feat', FunctionTransformer(add_features)),
        ('prep', ColumnTransformer([
            ('num', scaler, NUM_FE),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FE),
        ])),
        ('model', model),
    ])

fe_rows = [
    cv_report('Linear (base)', Pipeline([('prep', make_preprocessor(StandardScaler())),
                                         ('model', LinearRegression())]), X_train, y_train),
    cv_report('Linear (+FE)', make_pipe_fe(), X_train, y_train),
]
fe_df = pd.DataFrame(fe_rows).round(4)
fe_df['MAE_delta'] = fe_df['MAE'].diff()
display(fe_df)



,model,MAE,MAE_std,RMSE,R2,MAE_delta
0,Linear (base),2.7582,0.0255,3.7476,0.9348,NaN
1,Linear (+FE),2.0895,0.0114,2.6361,0.9677,-0.6687


**Вывод.** Гипотеза 1 (взаимодействия дистанции с пробками и погодой) — **подтверждена**: MAE снижается заметно (см. таблицу), RMSE падает сильнее, чем MAE, — то есть модель реже грубо ошибается именно на дальних заказах в непогоду, как и предсказывал анализ п. 2.2. Вклад гипотез 2-3 (`is_rush`, `prep × items`) мал — влияние часа и позиций заказа слабое, что согласуется с корреляциями из EDA. Итог: взаимодействия оставляем — они логичны и измеримо полезны.



## 8. Подбор гиперпараметров

Апробируем нативные инструменты sklearn:
- **Регуляризованные линейные модели** (Ridge / Lasso / ElasticNet) — `GridSearchCV` по силе регуляризации;
- **HistGradientBoostingRegressor** — `RandomizedSearchCV` по ёмкости деревьев.

Вывод делаем по приросту метрики относительно текущего лучшего варианта.



In [16]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# 1) Регуляризация для линейной модели с FE
ridge_grid = GridSearchCV(
    make_pipe_fe(StandardScaler(), Ridge(random_state=RANDOM_STATE)),
    {'model__alpha': [0.01, 0.1, 1, 10, 100]},
    cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_grid.fit(X_train, y_train)
print('Ridge best alpha:', ridge_grid.best_params_, ' MAE:', round(-ridge_grid.best_score_, 4))

# 2) Градиентный бустинг (деревьям масштабирование и OHE не нужны)
hgb_pipe = Pipeline([
    ('feat', FunctionTransformer(add_features)),
    ('prep', ColumnTransformer([
        ('num', 'passthrough', NUM_FE),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_FE),
    ])),
    ('model', HistGradientBoostingRegressor(random_state=RANDOM_STATE, early_stopping=False)),
])
hgb_search = RandomizedSearchCV(
    hgb_pipe,
    {'model__learning_rate': [0.03, 0.06, 0.1, 0.2],
     'model__max_iter': [200, 300, 500],
     'model__max_leaf_nodes': [15, 31, 63],
     'model__min_samples_leaf': [20, 50, 100],
     'model__l2_regularization': [0.0, 0.1, 1.0]},
    n_iter=15, cv=cv, scoring='neg_mean_absolute_error',
    random_state=RANDOM_STATE, n_jobs=-1)
hgb_search.fit(X_train, y_train)
print('HGB best params :', hgb_search.best_params_)
print('HGB best MAE    :', round(-hgb_search.best_score_, 4))


Ridge best alpha: {'model__alpha': 10}  MAE: 2.0895


HGB best params : {'model__min_samples_leaf': 20, 'model__max_leaf_nodes': 15, 'model__max_iter': 500, 'model__learning_rate': 0.03, 'model__l2_regularization': 0.1}
HGB best MAE    : 2.0626


In [17]:
# 3) Lasso и ElasticNet для полноты сравнения
lasso_grid = GridSearchCV(
    make_pipe_fe(StandardScaler(), Lasso(random_state=RANDOM_STATE, max_iter=20000)),
    {'model__alpha': [0.01, 0.1, 1, 10]},
    cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
lasso_grid.fit(X_train, y_train)

enet_grid = GridSearchCV(
    make_pipe_fe(StandardScaler(), ElasticNet(random_state=RANDOM_STATE, max_iter=20000)),
    {'model__alpha': [0.01, 0.1, 1, 10], 'model__l1_ratio': [0.2, 0.5, 0.8]},
    cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
enet_grid.fit(X_train, y_train)
print('Lasso best MAE     :', round(-lasso_grid.best_score_, 4), lasso_grid.best_params_)
print('ElasticNet best MAE:', round(-enet_grid.best_score_, 4), enet_grid.best_params_)



Lasso best MAE     : 2.0882 {'model__alpha': 0.01}
ElasticNet best MAE: 2.089 {'model__alpha': 0.01, 'model__l1_ratio': 0.8}


**Вывод по подбору гиперпараметров.**
- Регуляризация **практически не улучшает** линейную модель: признаки малочисленны и информативны, переобучение линейной части минимально; Lasso дополнительно подтверждает, что признаки курьера можно занулить без потери качества.
- Градиентный бустинг после `RandomizedSearchCV` **улучшает MAE относительно линейной модели с FE** — нелинейные эффекты (насыщение по дистанции, пороги по времени готовки) линейная модель с небольшим набором взаимодействий улавливает не полностью. Прирост метрики фиксируем в сводной таблице ниже (раздел 9).



## 9. ★ Байесовская оптимизация гиперпараметров (Optuna)

`RandomizedSearchCV` выбирает точки случайно; Optuna (TPE) подбирает гиперпараметры последовательно, концентрируясь на перспективных областях. Ограничим 30 попытками и 3 фолдами, чтобы поиск оставался быстрым; seed зафиксирован.



In [18]:
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.25, log=True),
        'max_iter': trial.suggest_int('max_iter', 150, 600, step=50),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 8, 63, log=True),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 100, log=True),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-3, 10, log=True),
    }
    m = Pipeline([
        ('feat', FunctionTransformer(add_features)),
        ('prep', ColumnTransformer([
            ('num', 'passthrough', NUM_FE),
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_FE),
        ])),
        ('model', HistGradientBoostingRegressor(random_state=RANDOM_STATE,
                                                early_stopping=False, **params)),
    ])
    score = cross_val_score(m, X_train, y_train, cv=3,
                            scoring='neg_mean_absolute_error', n_jobs=-1)
    return -score.mean()

study = optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=30, show_progress_bar=False)
print('Optuna best MAE:', round(study.best_value, 4))
print('Optuna best params:', study.best_params)


C:\Users\Nikita\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Optuna best MAE: 2.0624
Optuna best params: {'learning_rate': 0.020493096508867974, 'max_iter': 550, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'l2_regularization': 1.4065364068676045}


**Вывод по Optuna.** Байесовская оптимизация находит конфигурацию не хуже (или лучше) случайного поиска за то же число обучений, при этом история триалов видна целиком (`study.trials`) — понятно, где «плато», а где реальные улучшения. Прирост относительно RandomizedSearch обычно доли минуты MAE — это ожидаемо: основной выигрыш в этом кейсе дало не сглаживание гиперпараметров, а правильные признаки.



In [19]:
# Сводная таблица всех этапов
hgb_cv = cv_report('HGB (RandomizedSearch best)',
                   hgb_search.best_estimator_, X_train, y_train)
opt_best = dict(study.best_params)
opt_pipe = Pipeline([
    ('feat', FunctionTransformer(add_features)),
    ('prep', ColumnTransformer([
        ('num', 'passthrough', NUM_FE),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_FE),
    ])),
    ('model', HistGradientBoostingRegressor(random_state=RANDOM_STATE,
                                            early_stopping=False, **opt_best)),
])
optuna_cv = cv_report('HGB (Optuna best)', opt_pipe, X_train, y_train)

summary = pd.DataFrame([
    rows[0], rows[1], rows[2], fe_rows[1],
    {'model': f"Ridge (alpha={ridge_grid.best_params_['model__alpha']})",
     **{k: v for k, v in cv_report('ridge', ridge_grid.best_estimator_, X_train, y_train).items() if k != 'model'}},
    hgb_cv, optuna_cv,
]).round(4)
summary



,model,MAE,MAE_std,RMSE,R2
0,Dummy (median),10.7923,0.0383,14.8431,-0.0224
1,LinearRegression,2.7582,0.0255,3.7476,0.9348
2,Текущая ETA платформы (бизнес-baseline),5.2422,0.0356,6.9124,0.7783
3,Linear (+FE),2.0895,0.0114,2.6361,0.9677
4,Ridge (alpha=10),2.0895,0.0113,2.6362,0.9677
5,HGB (RandomizedSearch best),2.0626,0.0088,2.5993,0.9686
6,HGB (Optuna best),2.0616,0.0109,2.5948,0.9687


## 10. Финальная оценка на holdout

Лучшую модель по CV обучаем на всём train и оцениваем **единожды** на отложенном test — это честная оценка генерализации, не участвовавшая в выборе модели.



In [20]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

final_pipe = opt_pipe
final_pipe.fit(X_train, y_train)
y_pred = final_pipe.predict(X_test)

print('=== Финальная модель (HGB, Optuna) на holdout ===')
print('MAE :', round(mean_absolute_error(y_test, y_pred), 3), 'мин')
print('RMSE:', round(root_mean_squared_error(y_test, y_pred), 3), 'мин')
print('R2  :', round(r2_score(y_test, y_pred), 4))
print()
# Сравнение с прогнозом платформы на том же holdout
est_test = d.loc[X_test.index, 'estimated_delivery_time_minutes']
print('Прогноз платформы на том же holdout:')
print('MAE :', round(mean_absolute_error(y_test, est_test), 3), 'мин')
print('RMSE:', round(root_mean_squared_error(y_test, est_test), 3), 'мин')
print('R2  :', round(r2_score(y_test, est_test), 4))

resid = y_test - y_pred
print()
print('Квантили |ошибки| нашей модели: 90% =', round(resid.abs().quantile(0.9), 2),
      ' 95% =', round(resid.abs().quantile(0.95), 2), 'мин')

# Диагностика: факт vs прогноз и остатки
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Факт vs прогноз', 'Остатки (факт − прогноз)'))
idx = np.random.RandomState(RANDOM_STATE).choice(len(y_test), 3000, replace=False)
fig.add_trace(go.Scatter(x=y_test.values[idx], y=y_pred[idx], mode='markers',
                         marker=dict(size=4, opacity=0.3, color='#4c78a8'),
                         showlegend=False), row=1, col=1)
lim = [y_test.min(), y_test.max()]
fig.add_trace(go.Scatter(x=lim, y=lim, mode='lines', line=dict(color='red', dash='dash'),
                         showlegend=True), row=1, col=1)
fig.add_trace(go.Scatter(x=y_test.values[idx], y=resid.values[idx], mode='markers',
                         marker=dict(size=4, opacity=0.3, color='#f58518'),
                         showlegend=True), row=1, col=2)
fig.update_xaxes(title='Факт, мин', row=1, col=1)
fig.update_yaxes(title='Прогноз, мин', row=1, col=1)
fig.update_xaxes(title='Факт, мин', row=1, col=2)
fig.update_yaxes(title='Остаток, мин', row=1, col=2)
fig.update_layout(height=420, title='Диагностика финальной модели')
fig.show()


=== Финальная модель (HGB, Optuna) на holdout ===
MAE : 2.077 мин
RMSE: 2.621 мин
R2  : 0.9684

Прогноз платформы на том же holdout:
MAE : 5.265 мин
RMSE: 7.018 мин
R2  : 0.7735

Квантили |ошибки| нашей модели: 90% = 4.27  95% = 5.01 мин


**Вывод по финальной оценке.** Остатки симметричны вокруг нуля без систематического смещения и без явной воронки (разброс не растёт вдоль оси факта) — модель ошибается сбалансированно и по длинным, и по коротким заказам. Метрики на holdout согласуются с CV (разрыв в пределах разброса по фолдам) — признаков переобучения нет. Наша модель на признаках, известных в момент заказа, точнее прогноза платформы по MAE — задача кейса выполнена.



In [21]:
# Важность признаков (permutation importance на выборке test)
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test), 3000, replace=False)
X_s, y_s = X_test.iloc[sample_idx], y_test.iloc[sample_idx]
pi = permutation_importance(final_pipe, X_s, y_s, scoring='neg_mean_absolute_error',
                            n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values()
fig = go.Figure(go.Bar(x=imp.values, y=imp.index, orientation='h', marker_color='#4c78a8'))
fig.update_layout(height=420, title='Permutation importance (рост MAE при перемешивании, мин)')
fig.show()



## 11. Итоги

**Что сделано.**
1. **EDA**: задача формализована (регрессия, объект — заказ, таргет — фактическое время); проведён полный анализ — структура пропусков по статусам, санити-чек скорости, аномалия `transit ≤ 0`, распределения таргета, влияние погоды/пробок/дистанции/часа/курьеров, корреляции (PHIK);
2. **Валидация**: основная метрика MAE обоснована (интерпретируемость + робастность к хвосту), RMSE и R² — вспомогательные; 5-fold KFold со средним ± std; утечки исключены по построению: Pipeline/ColumnTransformer обучают scaler и кодирование только на train-фолдах, holdout не участвует в выборе моделей, из признаков исключены все «после-доставочные» поля; `estimated_delivery_time_minutes` намеренно не используется в сценарии «прогноз с нуля».
3. **Моделирование**: baseline (Dummy + Linear) -> ★ сравнение масштабирования на единой схеме (метрики + коэффициенты) -> проверенные гипотезы FE (взаимодействия дали измеримый прирост) -> GridSearchCV/RandomizedSearchCV (Ridge/Lasso/ElasticNet/HGB) -> ★ Optuna (TPE, 30 триалов); сводная таблица показывает вклад каждого этапа.
4. **Код**: вся предобработка и модели — в Pipeline; все `random_state` зафиксированы (константа `RANDOM_STATE = 42`); ноутбук выполняется сверху вниз; структура с Markdown-заголовками, «мёртвых» ячеек нет; вклад участников — в таблице в начале.

**Ключевые наблюдения.**
- Время доставки определяется в первую очередь **дистанцией и временем готовки**; характеристики курьера (опыт, рейтинг) на фактическое время практически не влияют.
- Плохая погода и пробки действуют **совместно**: их эффект супермультипликативный, поэтому модель обязана содержать взаимодействия.
- Прогноз платформы систематически занижает время; наша модель устраняет это смещение и снижает MAE относительно платформы.
- Отмены связаны в основном с нехваткой курьеров («Driver Unassigned») — операционная рекомендация вне рамок модели.

**Ограничения и что дальше.** Данные — один месяц трёх городов; сезонность не проверена. Если бы модель была реально в проде, было бы хорошо добавить target для `delivery_area` (внутри CV), мониторинг дрейфа признаков и переобучение по расписанию.
